# Smoke Test

Run all cells top-to-bottom. If everything prints **OK**, your environment is ready for the exercises.

Before you run: **Select Kernel** (top-right) → *Python Environments…* → pick the entry ending in `streaming/.venv/bin/python`.

## 1. Imports

Proves the venv is wired up and `confluent_kafka` is installed.

In [ ]:
from confluent_kafka import Producer, Consumer
from confluent_kafka.admin import AdminClient
import json, time

print('OK — confluent_kafka imported')

## 2. Connect to Redpanda and list topics

Reaches the broker on the Docker-internal address `redpanda:29092`. Should print at least the empty list (or whatever topics you've already created).

In [ ]:
BROKER = 'redpanda:29092'

admin = AdminClient({'bootstrap.servers': BROKER})
topics = admin.list_topics(timeout=5).topics
print(f'OK — connected to {BROKER}, {len(topics)} topic(s) so far: {sorted(topics)}')

## 3. Round-trip: produce one event and read it back

Writes a single message to a `smoke-test` topic and immediately consumes it.

In [ ]:
TOPIC = 'smoke-test'

producer = Producer({'bootstrap.servers': BROKER})
producer.produce(TOPIC, key=b'hello', value=json.dumps({'ts': time.time()}).encode())
producer.flush()
print(f'OK — produced 1 event to {TOPIC!r}')

consumer = Consumer({
    'bootstrap.servers': BROKER,
    'group.id':          f'smoke-{int(time.time())}',  # fresh group -> reads from earliest
    'auto.offset.reset': 'earliest',
})
consumer.subscribe([TOPIC])

for _ in range(10):              # poll up to ~5 seconds for a message
    msg = consumer.poll(0.5)
    if msg and not msg.error():
        print(f'OK — read back: key={msg.key()} value={msg.value()}')
        break
else:
    print('FAIL — no message received within 5 s')

consumer.close()

If all three sections printed **OK**, you're done — head to [`../02_demo/demo_produce.ipynb`](../02_demo/demo_produce.ipynb) and then [`../03_exercise/`](../03_exercise/).